# 04 Data-Source Ablation at 500 m

This notebook isolates the predictive information carried by the two main engineered data families used in the final Karachi electricity model.

It compares:

- **Remote sensing only** — Sentinel-2 band statistics, NDVI/NDBI, and VIIRS nighttime-light features.
- **Urban spatial only** — grid area, roads, POIs, population density, and building morphology.
- **All engineered features** — the final 36-feature tabular XGBoost model.

The experimental protocol is intentionally aligned with the final `03_tabular_and_cnn_evaluation.ipynb`:

- Random 5-fold outer CV;
- KMeans K=4 spatial leave-one-block-out outer CV using the existing upstream assignments;
- inner 3-fold CV;
- 30-iteration randomized XGBoost search;
- fold-wise median imputation;
- identical XGBoost search space;
- random seed = 42;
- spatial inner CV re-clusters the outer-training centroids into K=3 groups.

By default, the already-computed **All engineered features** result is reused from notebook 003 so only the two new single-source ablations need to be fitted. The notebook checks that the imported OOF fold membership matches the fixed outer splits before combining results.

This notebook writes to a **new isolated output directory** and does not overwrite notebook 003 or 004 outputs.


In [ ]:
# ============================================================
# Imports and global settings
# ============================================================

import gc
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
)
from sklearn.model_selection import (
    KFold,
    GroupKFold,
    RandomizedSearchCV,
)
from sklearn.pipeline import Pipeline

from xgboost import XGBRegressor

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
# ============================================================
# Paths, feature groups and experiment settings
# ============================================================

import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from project_config import RAW_DATA_DIR, WORK_DIR

PROJECT_DATA_DIR = WORK_DIR

FEATURE_DIR = (
    PROJECT_DATA_DIR
    / 'grid_size_selection'
    / 'features'
)

FEATURE_MATRIX_PATH = (
    FEATURE_DIR
    / 'feature_matrix_500m.csv'
)

SPATIAL_BLOCK_PATH = (
    PROJECT_DATA_DIR
    / 'model_tuning_500m'
    / 'spatial_block_assignments_500m.csv'
)

# Existing final 003 outputs.
SOURCE_003_DIR = (
    PROJECT_DATA_DIR
    / 'multimodal_fusion_500m'
)

SOURCE_003_FOLD_PATH = (
    SOURCE_003_DIR
    / 'multimodal_nested_fold_results.csv'
)

SOURCE_003_OOF_PATH = (
    SOURCE_003_DIR
    / 'multimodal_oof_predictions.csv'
)

SOURCE_003_SETTINGS_PATH = (
    SOURCE_003_DIR
    / '003_experiment_settings.json'
)

# New isolated output directory.
OUTPUT_DIR = (
    PROJECT_DATA_DIR
    / 'data_source_ablation_500m'
)

FIGURE_DIR = OUTPUT_DIR / 'figures'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COLUMN = 'elec_consumption'

REMOTE_SENSING_FEATURES = [
    'B02_mean', 'B02_std', 'B02_max',
    'B03_mean', 'B03_std', 'B03_max',
    'B04_mean', 'B04_std', 'B04_max',
    'B08_mean', 'B08_std', 'B08_max',
    'B11_mean', 'B11_std', 'B11_max',
    'NDVI', 'NDBI',
    'NTL_mean', 'NTL_max',
]

URBAN_SPATIAL_FEATURES = [
    'area_m2',
    'dist_major_road',
    'road_count',
    'road_length_major',
    'road_density',
    'major_road_ratio',
    'poi_economic_count',
    'poi_social_count',
    'poi_other_count',
    'poi_shannon',
    'dist_economic_poi',
    'dist_social_poi',
    'pop_density_km2',
    'building_count',
    'building_area_mean',
    'building_area_std',
    'building_coverage',
]

ALL_FEATURES = REMOTE_SENSING_FEATURES + URBAN_SPATIAL_FEATURES

assert len(REMOTE_SENSING_FEATURES) == 19
assert len(URBAN_SPATIAL_FEATURES) == 17
assert len(ALL_FEATURES) == 36
assert len(set(ALL_FEATURES)) == 36

RANDOM_FOLDS = 5
INNER_FOLDS = 3
SPATIAL_CLUSTERS = 4
SEARCH_ITERATIONS = 30

VALIDATION_TITLES = {
    'random_5fold': 'Random 5-fold CV',
    'spatial_kmeans_leave_one_out': 'Spatial KMeans K=4 CV',
}

# Recommended default: reuse the current 36-feature Tabular result from 003.
REUSE_ALL_FEATURES_FROM_003 = True

for required_path in [FEATURE_MATRIX_PATH, SPATIAL_BLOCK_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(
            f'Required upstream file not found: {required_path}'
        )

if REUSE_ALL_FEATURES_FROM_003:
    for required_path in [SOURCE_003_FOLD_PATH, SOURCE_003_OOF_PATH]:
        if not required_path.exists():
            raise FileNotFoundError(
                'REUSE_ALL_FEATURES_FROM_003=True, but a required '
                f'notebook-003 output is missing: {required_path}'
            )

print('Remote-sensing features:', len(REMOTE_SENSING_FEATURES))
print('Urban-spatial features:', len(URBAN_SPATIAL_FEATURES))
print('All engineered features:', len(ALL_FEATURES))
print('Search iterations:', SEARCH_ITERATIONS)
print('Output directory:', OUTPUT_DIR)


## 1. Load the final 500 m modelling matrix

The two feature families are mutually exclusive and together reproduce the final 36 engineered predictors used by the tabular XGBoost model.


In [ ]:
# ============================================================
# Load and validate the final modelling matrix
# ============================================================

tabular_df = pd.read_csv(FEATURE_MATRIX_PATH)
spatial_df = pd.read_csv(SPATIAL_BLOCK_PATH)

required_tabular_columns = [
    'grid_id',
    'centroid_x',
    'centroid_y',
    TARGET_COLUMN,
] + ALL_FEATURES

missing_tabular_columns = [
    column
    for column in required_tabular_columns
    if column not in tabular_df.columns
]

if missing_tabular_columns:
    raise KeyError(
        f'Missing required modelling columns: {missing_tabular_columns}'
    )

required_spatial_columns = ['grid_id', 'spatial_block']

missing_spatial_columns = [
    column
    for column in required_spatial_columns
    if column not in spatial_df.columns
]

if missing_spatial_columns:
    raise KeyError(
        f'Missing spatial-fold columns: {missing_spatial_columns}'
    )

if tabular_df['grid_id'].duplicated().any():
    raise ValueError('Duplicate grid_id values found in feature matrix.')

if spatial_df['grid_id'].duplicated().any():
    raise ValueError('Duplicate grid_id values found in spatial assignments.')

BASE_FRAME = (
    tabular_df[required_tabular_columns]
    .merge(
        spatial_df[required_spatial_columns],
        on='grid_id',
        how='inner',
        validate='one_to_one',
    )
    .sort_values('grid_id')
    .reset_index(drop=True)
)

if BASE_FRAME[TARGET_COLUMN].isna().any():
    raise ValueError('Target contains missing values.')

if BASE_FRAME['spatial_block'].isna().any():
    raise ValueError('Some modelling grids are missing spatial-block assignments.')

print('Modelling samples:', len(BASE_FRAME))
print(
    'Spatial blocks:',
    BASE_FRAME['spatial_block']
    .value_counts()
    .sort_index()
    .to_dict(),
)

display(
    BASE_FRAME[
        ['grid_id', TARGET_COLUMN, 'spatial_block']
    ].head()
)


In [ ]:
# ============================================================
# Optional protocol check against notebook 003 settings
# ============================================================

if SOURCE_003_SETTINGS_PATH.exists():
    with open(SOURCE_003_SETTINGS_PATH, 'r', encoding='utf-8') as file:
        source_003_settings = json.load(file)

    checks = {
        'random_outer_folds': RANDOM_FOLDS,
        'spatial_outer_blocks': SPATIAL_CLUSTERS,
        'inner_folds': INNER_FOLDS,
        'search_iterations': SEARCH_ITERATIONS,
    }

    print('Protocol check against 003_experiment_settings.json:')

    for key, expected in checks.items():
        found = source_003_settings.get(key, '<not recorded>')
        status = 'OK' if found == expected else 'CHECK'
        print(f'{status:>5} | {key}: 003={found} | 003b={expected}')
else:
    print(
        '003_experiment_settings.json not found. '
        'The notebook will continue using the protocol hard-coded above.'
    )


## 2. Fixed outer validation splits

These are reconstructed exactly as in the final notebook 003:

- Random: shuffled `KFold(n_splits=5, random_state=42)`.
- Spatial: leave one of the four pre-assigned KMeans blocks out at a time.


In [ ]:
# ============================================================
# Define fixed outer validation splits
# ============================================================

random_cv = KFold(
    n_splits=RANDOM_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

random_outer_splits = []

for fold, (train_idx, test_idx) in enumerate(
    random_cv.split(BASE_FRAME),
    start=1,
):
    random_outer_splits.append({
        'fold': fold,
        'train_index': train_idx,
        'test_index': test_idx,
    })

spatial_blocks_found = sorted(
    BASE_FRAME['spatial_block'].unique()
)

if len(spatial_blocks_found) != SPATIAL_CLUSTERS:
    raise ValueError(
        f'Expected {SPATIAL_CLUSTERS} spatial blocks, '
        f'found {len(spatial_blocks_found)}: {spatial_blocks_found}'
    )

spatial_outer_splits = []

for fold, block in enumerate(spatial_blocks_found, start=1):
    test_idx = np.where(
        BASE_FRAME['spatial_block'].to_numpy() == block
    )[0]
    train_idx = np.where(
        BASE_FRAME['spatial_block'].to_numpy() != block
    )[0]

    spatial_outer_splits.append({
        'fold': fold,
        'spatial_block': int(block),
        'train_index': train_idx,
        'test_index': test_idx,
    })

VALIDATION_SCHEMES = {
    'random_5fold': random_outer_splits,
    'spatial_kmeans_leave_one_out': spatial_outer_splits,
}

for validation_name, splits in VALIDATION_SCHEMES.items():
    print('\n' + VALIDATION_TITLES[validation_name])
    for split in splits:
        print(
            f"Fold {split['fold']}: "
            f"train={len(split['train_index'])}, "
            f"test={len(split['test_index'])}"
        )


## 3. XGBoost search space

The search space is copied from the final notebook 003 so differences between feature sets are not caused by changing the model family or tuning range.


In [ ]:
# ============================================================
# XGBoost model and shared search space
# ============================================================

def make_xgb_model():
    return XGBRegressor(
        objective='reg:squarederror',
        random_state=RANDOM_STATE,
        n_jobs=1,
        verbosity=0,
    )

XGB_SEARCH_SPACE = {
    'model__n_estimators': [300, 500, 800, 1200],
    'model__learning_rate': [0.01, 0.03, 0.05, 0.08],
    'model__max_depth': [2, 3, 4, 5, 6],
    'model__min_child_weight': [1, 3, 5, 10],
    'model__subsample': [0.60, 0.75, 0.90, 1.0],
    'model__colsample_bytree': [0.50, 0.70, 0.90, 1.0],
    'model__reg_alpha': [0, 0.01, 0.1, 0.5, 1],
    'model__reg_lambda': [1, 3, 5, 10, 20],
    'model__gamma': [0, 0.05, 0.10, 0.25],
}

print('XGBoost randomized-search iterations:', SEARCH_ITERATIONS)


In [ ]:
# ============================================================
# Leakage-safe feature pipeline
# ============================================================

def make_pipeline():
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', make_xgb_model()),
    ])


In [ ]:
# ============================================================
# Metrics and inner validation
# ============================================================

def regression_metrics(y_true, y_pred):
    return {
        'R2': r2_score(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred),
    }


def make_inner_cv(train_frame, validation_name, fold_seed):
    if validation_name == 'random_5fold':
        inner_cv = KFold(
            n_splits=INNER_FOLDS,
            shuffle=True,
            random_state=fold_seed,
        )
        return inner_cv, None

    coordinates = train_frame[
        ['centroid_x', 'centroid_y']
    ].to_numpy()

    inner_kmeans = KMeans(
        n_clusters=INNER_FOLDS,
        random_state=fold_seed,
        n_init=10,
    )

    groups = inner_kmeans.fit_predict(coordinates)

    inner_cv = GroupKFold(
        n_splits=INNER_FOLDS
    )

    return inner_cv, groups


## 4. General nested-CV runner

Each feature set is tuned independently inside the same outer-training folds. This avoids assuming that the hyperparameters selected for the 36-feature model are also optimal for a smaller feature family.


In [ ]:
# ============================================================
# General nested-CV runner for one engineered feature set
# ============================================================

def run_nested_feature_set(
    frame,
    splits,
    validation_name,
    feature_set,
    feature_columns,
):
    y_all = frame[TARGET_COLUMN].to_numpy(dtype=float)

    fold_rows = []
    prediction_frames = []

    for split in splits:
        fold = split['fold']
        train_idx = split['train_index']
        test_idx = split['test_index']

        train_frame = frame.iloc[train_idx].copy()
        test_frame = frame.iloc[test_idx].copy()

        X_train = train_frame[feature_columns]
        X_test = test_frame[feature_columns]

        y_train = y_all[train_idx]
        y_test = y_all[test_idx]

        fold_seed = RANDOM_STATE + fold

        inner_cv, inner_groups = make_inner_cv(
            train_frame=train_frame,
            validation_name=validation_name,
            fold_seed=fold_seed,
        )

        pipeline = make_pipeline()

        search = RandomizedSearchCV(
            estimator=pipeline,
            param_distributions=XGB_SEARCH_SPACE,
            n_iter=SEARCH_ITERATIONS,
            scoring='r2',
            cv=inner_cv,
            random_state=fold_seed,
            n_jobs=-1,
            refit=True,
            verbose=0,
        )

        fit_kwargs = {}
        if inner_groups is not None:
            fit_kwargs['groups'] = inner_groups

        search.fit(
            X_train,
            y_train,
            **fit_kwargs,
        )

        best_model = search.best_estimator_

        test_pred = best_model.predict(X_test)
        train_pred = best_model.predict(X_train)

        test_metrics = regression_metrics(y_test, test_pred)
        train_r2 = r2_score(y_train, train_pred)

        spatial_block = split.get('spatial_block', np.nan)

        clean_best_params = {
            key.replace('model__', ''): value
            for key, value in search.best_params_.items()
        }

        fold_rows.append({
            'validation': validation_name,
            'feature_set': feature_set,
            'target_mode': 'raw',
            'fold': fold,
            'spatial_block': spatial_block,
            'n_train': len(train_idx),
            'n_test': len(test_idx),
            'n_features': len(feature_columns),
            'R2': test_metrics['R2'],
            'RMSE': test_metrics['RMSE'],
            'MAE': test_metrics['MAE'],
            'train_R2': train_r2,
            'inner_best_R2_fit_scale': search.best_score_,
            'best_params': json.dumps(clean_best_params, sort_keys=True),
        })

        prediction_frames.append(
            pd.DataFrame({
                'validation': validation_name,
                'feature_set': feature_set,
                'target_mode': 'raw',
                'fold': fold,
                'spatial_block': spatial_block,
                'grid_id': test_frame['grid_id'].to_numpy(),
                'observed': y_test,
                'predicted': test_pred,
                'residual': y_test - test_pred,
            })
        )

        print(
            f'{validation_name} | {feature_set} | fold {fold} | '
            f"R²={test_metrics['R2']:.4f} | "
            f"RMSE={test_metrics['RMSE']:.2f} | "
            f"MAE={test_metrics['MAE']:.2f}"
        )

        del search, best_model, X_train, X_test, train_pred, test_pred
        gc.collect()

    fold_results = pd.DataFrame(fold_rows)
    predictions = pd.concat(prediction_frames, ignore_index=True)

    return fold_results, predictions


## 5. Run the two new data-source ablations

Only **Remote sensing only** and **Urban spatial only** are fitted here by default.


In [ ]:
# ============================================================
# Run the two new single-source ablations
# ============================================================

NEW_FEATURE_SETS = {
    'Remote sensing only': REMOTE_SENSING_FEATURES,
    'Urban spatial only': URBAN_SPATIAL_FEATURES,
}

new_fold_frames = []
new_prediction_frames = []

for feature_set, feature_columns in NEW_FEATURE_SETS.items():
    for validation_name, splits in VALIDATION_SCHEMES.items():
        print('\n' + '=' * 78)
        print(VALIDATION_TITLES[validation_name], '|', feature_set)

        fold_result, predictions = run_nested_feature_set(
            frame=BASE_FRAME,
            splits=splits,
            validation_name=validation_name,
            feature_set=feature_set,
            feature_columns=feature_columns,
        )

        new_fold_frames.append(fold_result)
        new_prediction_frames.append(predictions)

new_fold_results = pd.concat(new_fold_frames, ignore_index=True)
new_oof_predictions = pd.concat(new_prediction_frames, ignore_index=True)

display(new_fold_results.round(4))


## 6. Add the 36-feature reference result

Default behaviour is to reuse the final `Tabular` / raw-target result already produced by notebook 003. Before accepting it, this section verifies that each imported OOF fold contains exactly the same grid IDs as the fixed outer split reconstructed above.

If you intentionally want to refit all 36 features again, set `REUSE_ALL_FEATURES_FROM_003 = False` in the settings cell and rerun the notebook.


In [ ]:
# ============================================================
# Validate imported 003 OOF fold membership
# ============================================================

def expected_test_grid_ids(frame, validation_name, fold):
    split_list = VALIDATION_SCHEMES[validation_name]
    matching = [
        split
        for split in split_list
        if split['fold'] == fold
    ]

    if len(matching) != 1:
        raise ValueError(
            f'Could not uniquely resolve {validation_name} fold {fold}.'
        )

    test_idx = matching[0]['test_index']
    return set(frame.iloc[test_idx]['grid_id'].tolist())


def validate_imported_oof_membership(imported_oof):
    problems = []

    for (validation_name, fold), group in imported_oof.groupby(
        ['validation', 'fold']
    ):
        expected_ids = expected_test_grid_ids(
            BASE_FRAME,
            validation_name,
            int(fold),
        )
        imported_ids = set(group['grid_id'].tolist())

        if imported_ids != expected_ids:
            problems.append(
                (
                    validation_name,
                    int(fold),
                    len(expected_ids),
                    len(imported_ids),
                )
            )

    if problems:
        raise ValueError(
            'Imported notebook-003 OOF fold membership does not match '
            'the fixed 003b splits. Set REUSE_ALL_FEATURES_FROM_003=False '
            f'and rerun. Problems: {problems}'
        )

    print('Imported notebook-003 OOF membership matches all fixed outer folds.')


In [ ]:
# ============================================================
# Reuse or refit the 36-feature reference model
# ============================================================

if REUSE_ALL_FEATURES_FROM_003:
    source_fold = pd.read_csv(SOURCE_003_FOLD_PATH)
    source_oof = pd.read_csv(SOURCE_003_OOF_PATH)

    source_fold = source_fold[
        source_fold['feature_set'] == 'Tabular'
    ].copy()

    source_oof = source_oof[
        source_oof['feature_set'] == 'Tabular'
    ].copy()

    if 'target_mode' in source_fold.columns:
        source_fold = source_fold[
            source_fold['target_mode'] == 'raw'
        ].copy()

    if 'target_mode' in source_oof.columns:
        source_oof = source_oof[
            source_oof['target_mode'] == 'raw'
        ].copy()

    if source_fold.empty:
        raise ValueError(
            'No Tabular/raw fold results were found in notebook-003 outputs.'
        )

    if source_oof.empty:
        raise ValueError(
            'No Tabular/raw OOF predictions were found in notebook-003 outputs.'
        )

    validate_imported_oof_membership(source_oof)

    all_feature_fold_results = source_fold.copy()
    all_feature_oof_predictions = source_oof.copy()

    all_feature_fold_results['feature_set'] = 'All engineered features'
    all_feature_oof_predictions['feature_set'] = 'All engineered features'

    all_feature_fold_results['n_features'] = len(ALL_FEATURES)
    all_feature_fold_results['target_mode'] = 'raw'
    all_feature_oof_predictions['target_mode'] = 'raw'

    if 'residual' not in all_feature_oof_predictions.columns:
        all_feature_oof_predictions['residual'] = (
            all_feature_oof_predictions['observed']
            - all_feature_oof_predictions['predicted']
        )

    print('Reused the final 36-feature Tabular result from notebook 003.')

else:
    all_fold_frames = []
    all_prediction_frames = []

    for validation_name, splits in VALIDATION_SCHEMES.items():
        print('\n' + '=' * 78)
        print(
            VALIDATION_TITLES[validation_name],
            '| All engineered features',
        )

        fold_result, predictions = run_nested_feature_set(
            frame=BASE_FRAME,
            splits=splits,
            validation_name=validation_name,
            feature_set='All engineered features',
            feature_columns=ALL_FEATURES,
        )

        all_fold_frames.append(fold_result)
        all_prediction_frames.append(predictions)

    all_feature_fold_results = pd.concat(
        all_fold_frames,
        ignore_index=True,
    )

    all_feature_oof_predictions = pd.concat(
        all_prediction_frames,
        ignore_index=True,
    )


In [ ]:
# ============================================================
# Combine all three feature sets
# ============================================================

ablation_fold_results = pd.concat(
    [new_fold_results, all_feature_fold_results],
    ignore_index=True,
    sort=False,
)

ablation_oof_predictions = pd.concat(
    [new_oof_predictions, all_feature_oof_predictions],
    ignore_index=True,
    sort=False,
)

FEATURE_SET_ORDER = [
    'Remote sensing only',
    'Urban spatial only',
    'All engineered features',
]

ablation_fold_results['feature_set'] = pd.Categorical(
    ablation_fold_results['feature_set'],
    categories=FEATURE_SET_ORDER,
    ordered=True,
)

ablation_oof_predictions['feature_set'] = pd.Categorical(
    ablation_oof_predictions['feature_set'],
    categories=FEATURE_SET_ORDER,
    ordered=True,
)

ablation_fold_results = (
    ablation_fold_results
    .sort_values(['validation', 'feature_set', 'fold'])
    .reset_index(drop=True)
)

ablation_oof_predictions = (
    ablation_oof_predictions
    .sort_values(['validation', 'feature_set', 'grid_id'])
    .reset_index(drop=True)
)

print('Combined fold-result rows:', len(ablation_fold_results))
print('Combined OOF rows:', len(ablation_oof_predictions))


## 7. Summary metrics

`R2_std`, `RMSE_std`, and `MAE_std` use **sample standard deviation (`ddof=1`)** so the reported uncertainty is consistent across all three feature sets.


In [ ]:
# ============================================================
# Summarise fold-level and pooled OOF performance
# ============================================================

def summarise_results(fold_results, oof_predictions):
    summary = (
        fold_results
        .groupby(
            ['validation', 'feature_set'],
            observed=True,
            as_index=False,
        )
        .agg(
            n_folds=('fold', 'count'),
            R2_mean=('R2', 'mean'),
            R2_std=('R2', lambda values: values.std(ddof=1)),
            RMSE_mean=('RMSE', 'mean'),
            RMSE_std=('RMSE', lambda values: values.std(ddof=1)),
            MAE_mean=('MAE', 'mean'),
            MAE_std=('MAE', lambda values: values.std(ddof=1)),
            train_R2_mean=('train_R2', 'mean'),
        )
    )

    oof_rows = []

    for (validation_name, feature_set), group in oof_predictions.groupby(
        ['validation', 'feature_set'],
        observed=True,
    ):
        metrics = regression_metrics(
            group['observed'],
            group['predicted'],
        )

        oof_rows.append({
            'validation': validation_name,
            'feature_set': feature_set,
            'n_predictions': len(group),
            'OOF_R2': metrics['R2'],
            'OOF_RMSE': metrics['RMSE'],
            'OOF_MAE': metrics['MAE'],
        })

    summary = summary.merge(
        pd.DataFrame(oof_rows),
        on=['validation', 'feature_set'],
        how='left',
    )

    summary['R2_generalisation_gap'] = (
        summary['train_R2_mean']
        - summary['R2_mean']
    )

    summary['feature_set'] = pd.Categorical(
        summary['feature_set'],
        categories=FEATURE_SET_ORDER,
        ordered=True,
    )

    return (
        summary
        .sort_values(['validation', 'feature_set'])
        .reset_index(drop=True)
    )

ablation_summary = summarise_results(
    ablation_fold_results,
    ablation_oof_predictions,
)

display(ablation_summary.round(4))


## 8. Complementarity diagnostics

These deltas are descriptive ablation contrasts, not additive causal contributions.

- `All minus Remote` asks how much performance changes when urban-spatial information is added to the remote-sensing feature family.
- `All minus Urban` asks how much performance changes when engineered remote-sensing information is added to the urban-spatial feature family.
- `All minus Best single source` measures the advantage of the combined engineered feature set over whichever single-source family performs better.


In [ ]:
# ============================================================
# Build compact data-source complementarity table
# ============================================================

comparison_rows = []

for validation_name in VALIDATION_SCHEMES:
    subset = (
        ablation_summary[
            ablation_summary['validation'] == validation_name
        ]
        .set_index('feature_set')
    )

    remote = subset.loc['Remote sensing only']
    urban = subset.loc['Urban spatial only']
    combined = subset.loc['All engineered features']

    best_single_r2 = max(remote['R2_mean'], urban['R2_mean'])
    best_single_oof = max(remote['OOF_R2'], urban['OOF_R2'])

    comparison_rows.append({
        'validation': validation_name,
        'All_minus_Remote_R2_mean': combined['R2_mean'] - remote['R2_mean'],
        'All_minus_Urban_R2_mean': combined['R2_mean'] - urban['R2_mean'],
        'All_minus_best_single_R2_mean': combined['R2_mean'] - best_single_r2,
        'All_minus_Remote_OOF_R2': combined['OOF_R2'] - remote['OOF_R2'],
        'All_minus_Urban_OOF_R2': combined['OOF_R2'] - urban['OOF_R2'],
        'All_minus_best_single_OOF_R2': combined['OOF_R2'] - best_single_oof,
    })

complementarity_table = pd.DataFrame(comparison_rows)

display(complementarity_table.round(4))


## 9. Figures

The bars show mean outer-fold R²; error bars show ±1 sample SD. Random and spatial validation are plotted separately because their R² scales can differ substantially.


In [ ]:
# ============================================================
# Plot mean outer-fold R² for each validation scheme
# ============================================================

for validation_name in VALIDATION_SCHEMES:
    plot_df = (
        ablation_summary[
            ablation_summary['validation'] == validation_name
        ]
        .copy()
        .sort_values('feature_set')
    )

    x = np.arange(len(plot_df))

    fig, ax = plt.subplots(figsize=(8, 6))

    ax.bar(
        x,
        plot_df['R2_mean'].to_numpy(),
        yerr=plot_df['R2_std'].to_numpy(),
        capsize=5,
    )

    ax.axhline(0, linewidth=0.8)

    ax.set_xticks(x)
    ax.set_xticklabels(
        plot_df['feature_set'].astype(str),
        rotation=15,
        ha='right',
    )

    ax.set_ylabel('Mean outer-fold R²')
    ax.set_title(
        VALIDATION_TITLES[validation_name]
        + ' — Data-source ablation'
    )

    fig.tight_layout()

    figure_path = (
        FIGURE_DIR
        / ('data_source_ablation_' + validation_name + '_r2.png')
    )

    fig.savefig(
        figure_path,
        dpi=300,
        bbox_inches='tight',
    )

    plt.show()
    print('Saved:', figure_path)


In [ ]:
# ============================================================
# Optional compact OOF R² plot
# ============================================================

for validation_name in VALIDATION_SCHEMES:
    plot_df = (
        ablation_summary[
            ablation_summary['validation'] == validation_name
        ]
        .copy()
        .sort_values('feature_set')
    )

    x = np.arange(len(plot_df))

    fig, ax = plt.subplots(figsize=(8, 6))

    ax.bar(
        x,
        plot_df['OOF_R2'].to_numpy(),
    )

    ax.axhline(0, linewidth=0.8)

    ax.set_xticks(x)
    ax.set_xticklabels(
        plot_df['feature_set'].astype(str),
        rotation=15,
        ha='right',
    )

    ax.set_ylabel('Pooled OOF R²')
    ax.set_title(
        VALIDATION_TITLES[validation_name]
        + ' — Pooled OOF performance'
    )

    fig.tight_layout()

    figure_path = (
        FIGURE_DIR
        / ('data_source_ablation_' + validation_name + '_oof_r2.png')
    )

    fig.savefig(
        figure_path,
        dpi=300,
        bbox_inches='tight',
    )

    plt.show()
    print('Saved:', figure_path)


## 10. Save outputs

These files are intentionally separate from notebook 003 and notebook 004.


In [ ]:
# ============================================================
# Save data-source ablation outputs
# ============================================================

FOLD_OUTPUT_PATH = (
    OUTPUT_DIR
    / 'data_source_ablation_nested_fold_results.csv'
)

OOF_OUTPUT_PATH = (
    OUTPUT_DIR
    / 'data_source_ablation_oof_predictions.csv'
)

SUMMARY_OUTPUT_PATH = (
    OUTPUT_DIR
    / 'data_source_ablation_summary.csv'
)

COMPLEMENTARITY_OUTPUT_PATH = (
    OUTPUT_DIR
    / 'data_source_ablation_complementarity.csv'
)

SETTINGS_OUTPUT_PATH = (
    OUTPUT_DIR
    / '003b_experiment_settings.json'
)

ablation_fold_results.to_csv(FOLD_OUTPUT_PATH, index=False)
ablation_oof_predictions.to_csv(OOF_OUTPUT_PATH, index=False)
ablation_summary.to_csv(SUMMARY_OUTPUT_PATH, index=False)
complementarity_table.to_csv(COMPLEMENTARITY_OUTPUT_PATH, index=False)

settings = {
    'notebook': '04_data_source_ablation.ipynb',
    'random_state': RANDOM_STATE,
    'target_column': TARGET_COLUMN,
    'remote_sensing_feature_count': len(REMOTE_SENSING_FEATURES),
    'remote_sensing_features': REMOTE_SENSING_FEATURES,
    'urban_spatial_feature_count': len(URBAN_SPATIAL_FEATURES),
    'urban_spatial_features': URBAN_SPATIAL_FEATURES,
    'all_feature_count': len(ALL_FEATURES),
    'all_features': ALL_FEATURES,
    'random_outer_folds': RANDOM_FOLDS,
    'spatial_outer_blocks': SPATIAL_CLUSTERS,
    'inner_folds': INNER_FOLDS,
    'search_iterations': SEARCH_ITERATIONS,
    'reuse_all_features_from_003': REUSE_ALL_FEATURES_FROM_003,
    'source_003_directory': str(SOURCE_003_DIR),
    'output_directory': str(OUTPUT_DIR),
    'fold_sd_ddof': 1,
}

with open(SETTINGS_OUTPUT_PATH, 'w', encoding='utf-8') as file:
    json.dump(settings, file, indent=2)

print('Saved:')
print(FOLD_OUTPUT_PATH)
print(OOF_OUTPUT_PATH)
print(SUMMARY_OUTPUT_PATH)
print(COMPLEMENTARITY_OUTPUT_PATH)
print(SETTINGS_OUTPUT_PATH)


## Interpretation checklist

Use the results in the following order:

1. Compare **Remote sensing only** and **Urban spatial only** under the same validation scheme to identify which family carries the stronger standalone predictive signal.
2. Compare each single-source family with **All engineered features** to assess whether combining the two information sources improves prediction.
3. Do not interpret the R² differences as causal or additive feature contributions.
4. Compare both mean outer-fold R² ± SD and pooled OOF metrics.
5. Give spatial-CV results equal interpretive weight: a feature family that performs well only under Random CV should not be described as spatially robust.
6. Keep the existing CNN experiment separate: this notebook asks **which engineered data families contain predictive information**, while notebook 003 asks whether **CNN-derived visual representations add information beyond the combined engineered predictors**.
